In [ ]:
!pip install -q torch torchvision opacus scikit-learn scipy matplotlib

In [ ]:
import torch, torchvision, numpy as np, random, math, copy
from torch import nn
from torchvision import transforms
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import sys
sys.setrecursionlimit(10000)

# Reproducibility (global seed for split)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# CIFAR‑10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

full_train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                              download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)

# Take 5000 samples from the training set
np.random.seed(SEED)
all_indices = np.arange(len(full_train_set))
np.random.shuffle(all_indices)
subset_indices = all_indices[:5000]
train_subset = torch.utils.data.Subset(full_train_set, subset_indices)

# ---------- Fixed public/private split (1000 public, 4000 private) ----------
np.random.seed(SEED)   # re‑seed to make split deterministic
indices = np.arange(len(train_subset))
np.random.shuffle(indices)
public_indices = indices[:1000]
private_indices = indices[1000:]

public_set = torch.utils.data.Subset(train_subset, public_indices)
private_set = torch.utils.data.Subset(train_subset, private_indices)

public_loader = torch.utils.data.DataLoader(public_set, batch_size=256,
                                            shuffle=False, num_workers=2)
private_loader = torch.utils.data.DataLoader(private_set, batch_size=256,
                                             shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256,
                                          shuffle=False, num_workers=2)

print(f"Public: {len(public_set)}   Private: {len(private_set)}   Test: {len(test_set)}")

In [ ]:
# ---------- Feature extractor ----------
resnet18 = torchvision.models.resnet18(pretrained=True)
feature_extractor = nn.Sequential(*list(resnet18.children())[:-1])
feature_extractor.eval()
for p in feature_extractor.parameters():
    p.requires_grad = False
feature_extractor.to(device)

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            out = feature_extractor(imgs)
            out = out.view(out.size(0), -1)
            feats.append(out.cpu().numpy())
            labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)

print("Extracting features …")
X_public,  y_public  = extract_features(public_loader)
X_private, y_private = extract_features(private_loader)
X_test,    y_test    = extract_features(test_loader)

# ---------- Scale to [-1,1] using private‑set statistics ----------
def fit_scaler(arr):
    minv = arr.min(axis=0, keepdims=True)
    maxv = arr.max(axis=0, keepdims=True)
    rng = maxv - minv + 1e-8
    return minv, maxv, rng

def apply_scaler(arr, minv, maxv, rng):
    return 2 * (arr - minv) / rng - 1.0

minv, maxv, rng = fit_scaler(X_private)
X_public  = apply_scaler(X_public,  minv, maxv, rng)
X_private = apply_scaler(X_private, minv, maxv, rng)
X_test    = apply_scaler(X_test,    minv, maxv, rng)

print(f"Shapes: public={X_public.shape}, private={X_private.shape}, test={X_test.shape}")

In [ ]:
def ball_variance(points):
    if len(points) == 0:
        return 0.0
    centroid = points.mean(axis=0)
    dists = np.linalg.norm(points - centroid, axis=1)
    return np.mean(dists**2)

def split_ball(points, labels, theta, min_samples, depth=0, max_depth=50):
    if depth > max_depth:
        center = points.mean(axis=0)
        radius = np.max(np.linalg.norm(points - center, axis=1)) if len(points) > 0 else 0.0
        values, counts = np.unique(labels, return_counts=True)
        label = values[np.argmax(counts)]
        purity = counts.max() / len(labels)
        return [{'center': center, 'radius': radius, 'label': int(label),
                 'purity': float(purity), 'size': int(len(points))}]

    if len(points) < min_samples or ball_variance(points) <= theta:
        center = points.mean(axis=0)
        radius = np.max(np.linalg.norm(points - center, axis=1)) if len(points) > 0 else 0.0
        values, counts = np.unique(labels, return_counts=True)
        label = values[np.argmax(counts)]
        purity = counts.max() / len(labels)
        return [{'center': center, 'radius': radius, 'label': int(label),
                 'purity': float(purity), 'size': int(len(points))}]

    var_per_dim = np.var(points, axis=0)
    split_dim = np.argmax(var_per_dim)
    median = np.median(points[:, split_dim])
    left_idx  = points[:, split_dim] <= median
    right_idx = points[:, split_dim] > median

    if np.sum(left_idx) == 0 or np.sum(right_idx) == 0:
        center = points.mean(axis=0)
        radius = np.max(np.linalg.norm(points - center, axis=1)) if len(points) > 0 else 0.0
        values, counts = np.unique(labels, return_counts=True)
        label = values[np.argmax(counts)]
        purity = counts.max() / len(labels)
        return [{'center': center, 'radius': radius, 'label': int(label),
                 'purity': float(purity), 'size': int(len(points))}]

    left_balls  = split_ball(points[left_idx],  labels[left_idx],
                             theta, min_samples, depth+1, max_depth)
    right_balls = split_ball(points[right_idx], labels[right_idx],
                             theta, min_samples, depth+1, max_depth)
    return left_balls + right_balls

def generate_granular_balls(X, y, theta=0.1, min_samples=50):
    balls = split_ball(X, y, theta, min_samples)
    print(f"Generated {len(balls)} granular balls (θ={theta}, min_samples={min_samples})")
    return balls

In [ ]:
def dp_noise_ball(ball, epsilon_ball, delta_ball=1e-7):
    dim = len(ball['center'])
    n_points = ball['size']
    sensitivity_center = 2.0 * math.sqrt(dim) / n_points
    sigma = sensitivity_center * math.sqrt(2 * math.log(1.25 / delta_ball)) / epsilon_ball
    noise_center = np.random.normal(0, sigma, size=dim)
    center_dp = ball['center'] + noise_center

    b = sensitivity_center / epsilon_ball
    radius_dp = ball['radius'] + np.random.laplace(0, b)
    radius_dp = max(radius_dp, 0.0)

    return {
        'center': center_dp,
        'radius': radius_dp,
        'label': ball['label'],
        'purity': ball['purity'],
        'size': ball['size']
    }

def apply_dp_to_balls(balls, epsilon_total, delta_total=1e-5):
    n = len(balls)
    eps_i = epsilon_total / n
    delta_i = delta_total / n
    return [dp_noise_ball(b, eps_i, delta_i) for b in balls]

In [ ]:
def predict_probabilistic(balls, x, temperature=1.0):
    centers = np.stack([b['center'] for b in balls])
    purities = np.array([b['purity'] for b in balls])
    dists = np.linalg.norm(centers - x, axis=1)
    dists_stable = dists - np.max(dists)  # stabilise
    weights = np.exp(np.clip(-dists_stable / temperature, -700, 700)) * purities
    if weights.sum() == 0:
        weights = np.ones_like(weights)
    class_scores = np.zeros(10)
    for i, b in enumerate(balls):
        class_scores[b['label']] += weights[i]
    return np.argmax(class_scores)

def evaluate(balls, X, y, temperature=1.0):
    preds = np.array([predict_probabilistic(balls, x, temperature) for x in X])
    acc = accuracy_score(y, preds)
    return acc, preds

In [ ]:
# New hyperparameter search: target 2-10 balls
min_samples_candidates = [400, 500, 600, 700, 800]   # yields 2-8 balls
theta = 0.1
best_acc = 0
best_params = {}
for min_s in min_samples_candidates:
    balls = generate_granular_balls(X_public, y_public, theta=theta, min_samples=min_s)
    acc, _ = evaluate(balls, X_test, y_test, temperature=1.0)
    print(f"min_samples={min_s:3d}: {len(balls)} balls, test acc={acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_params = {'min_samples': min_s, 'theta': theta}
print(f"\nBest hyperparameters: {best_params} with non‑private acc={best_acc:.4f}")

In [ ]:
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.accountants.utils import get_noise_multiplier

class FCNet(nn.Module):
    def __init__(self, input_dim=512, hidden=256, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x):
        return self.net(x)

def make_feature_loader(X, y, batch_size=256, shuffle=True):
    dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(y).long()
    )
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

private_feat_loader = make_feature_loader(X_private, y_private, shuffle=True)
test_feat_loader    = make_feature_loader(X_test, y_test, shuffle=False)

def train_dp_sgd(target_epsilon=5.0, epochs=4, delta=1e-5):
    model = FCNet(input_dim=X_private.shape[1]).to(device)
    model = ModuleValidator.fix(model)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    batch_size = 256
    sample_rate = batch_size / len(X_private)
    steps = epochs * (len(X_private) // batch_size)
    noise_multiplier = get_noise_multiplier(
        target_epsilon=target_epsilon,
        target_delta=delta,
        sample_rate=sample_rate,
        steps=steps,
        accountant="rdp"
    )

    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader_dp = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=private_feat_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=1.0,
    )
    model.train()
    for epoch in range(epochs):
        for xb, yb in train_loader_dp:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    actual_epsilon = privacy_engine.get_epsilon(delta)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in test_feat_loader:
            xb = xb.to(device)
            preds = model(xb).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
    test_acc = accuracy_score(y_test, np.concatenate(all_preds))
    return model, actual_epsilon, test_acc

In [ ]:
import scipy.stats as st

# Use the best hyperparameters (min_samples giving ~2 balls)
best_min_samples = 800   # or whatever gave ~2 balls in your search

# Build public balls once (deterministic)
public_balls = generate_granular_balls(
    X_public,
    y_public,
    theta=0.1,
    min_samples=best_min_samples
)

print(f"Using {len(public_balls)} public balls for DP-GbC")

SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]
all_results = []

for run_seed in SEEDS:
    print(f"\n===== Seed {run_seed} =====")

    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    torch.cuda.manual_seed_all(run_seed)

    # DP-GbC with fixed public balls (2-ball setup)
    dp_balls = apply_dp_to_balls(
        public_balls,
        epsilon_total=5.0,
        delta_total=1e-5
    )
    acc_gbc, _ = evaluate(dp_balls, X_test, y_test, temperature=1.0)

    # DP-SGD (unchanged)
    _, eps_sgd, acc_sgd = train_dp_sgd(target_epsilon=5.0, epochs=4)

    all_results.append({
        'seed': run_seed,
        'acc_gbc': acc_gbc,
        'acc_sgd': acc_sgd,
        'eps_sgd': eps_sgd
    })


# ---------- Bootstrap confidence intervals ----------
def bootstrap_ci(data, n_bootstrap=10000):
    means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        means.append(np.mean(sample))
    return np.percentile(means, 2.5), np.percentile(means, 97.5)


gbc_accs = [r['acc_gbc'] for r in all_results]
sgd_accs = [r['acc_sgd'] for r in all_results]
eps_sgds = [r['eps_sgd'] for r in all_results]

gbc_mean = np.mean(gbc_accs)
sgd_mean = np.mean(sgd_accs)
eps_mean = np.mean(eps_sgds)

gbc_ci = bootstrap_ci(gbc_accs)
sgd_ci = bootstrap_ci(sgd_accs)
eps_ci = bootstrap_ci(eps_sgds)

print("\n" + "=" * 60)
print("Final Results (10 seeds)")
print(f"DP-GbC (ε=5.0):     Acc = {gbc_mean*100:.2f}%  (95% CI: {gbc_ci[0]*100:.2f}–{gbc_ci[1]*100:.2f}%)")
print(f"DP-SGD (ε≈{eps_mean:.2f}): Acc = {sgd_mean*100:.2f}%  (95% CI: {sgd_ci[0]*100:.2f}–{sgd_ci[1]*100:.2f}%)")

In [ ]:
import scipy.stats as st

SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]   # 10 seeds
all_results = []

# Fixed public balls (deterministic because public data is fixed)
public_balls = generate_granular_balls(X_public, y_public, theta=0.1, min_samples=50)
print(f"Number of public balls: {len(public_balls)}")

for run_seed in SEEDS:
    print(f"\n===== Seed {run_seed} =====")
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    torch.cuda.manual_seed_all(run_seed)

    # --- DP‑GbC: add noise to the public balls ---
    dp_balls = apply_dp_to_balls(public_balls, epsilon_total=5.0, delta_total=1e-5)
    acc_gbc, _ = evaluate(dp_balls, X_test, y_test, temperature=1.0)

    # --- DP‑SGD (target ε=5.0) on private data ---
    _, eps_sgd, acc_sgd = train_dp_sgd(target_epsilon=5.0, epochs=4)

    all_results.append({
        'seed': run_seed,
        'acc_gbc': acc_gbc,
        'acc_sgd': acc_sgd,
        'eps_sgd': eps_sgd
    })

# ---------- Results with bootstrap confidence intervals ----------
def bootstrap_ci(data, n_bootstrap=10000):
    means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        means.append(np.mean(sample))
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

gbc_accs = [r['acc_gbc'] for r in all_results]
sgd_accs = [r['acc_sgd'] for r in all_results]
eps_sgds = [r['eps_sgd'] for r in all_results]

gbc_mean, gbc_ci = np.mean(gbc_accs), bootstrap_ci(gbc_accs)
sgd_mean, sgd_ci = np.mean(sgd_accs), bootstrap_ci(sgd_accs)
eps_mean, eps_ci = np.mean(eps_sgds), bootstrap_ci(eps_sgds)

print("\n" + "="*70)
print("Final Results (10 seeds)")
print(f"DP‑GbC (ε=5.0):     Acc = {gbc_mean*100:.2f}%  (95% CI: {gbc_ci[0]*100:.2f}–{gbc_ci[1]*100:.2f}%)")
print(f"DP‑SGD (ε≈{eps_mean:.2f}): Acc = {sgd_mean*100:.2f}%  (95% CI: {sgd_ci[0]*100:.2f}–{sgd_ci[1]*100:.2f}%)")

In [ ]:
# ============================================================
# Additional analyses: temperature tuning, epsilon sweeps, MIA
# ============================================================
print("\n" + "="*70)
print("ADDITIONAL ANALYSES (temperature tuning, epsilon sweeps, MIA)")
print("="*70)

# ------------------------------------------------------------------
# 1. Temperature tuning for probabilistic prediction
#    (using public set as validation – no privacy cost)
# ------------------------------------------------------------------
temp_candidates = [0.5, 1.0, 2.0, 5.0, 10.0]
best_temp = 1.0
best_acc_temp = 0.0

print("\n--- Tuning temperature on public set ---")
public_balls = generate_granular_balls(X_public, y_public,
                                       theta=best_params['theta'],
                                       min_samples=best_params['min_samples'])
for t in temp_candidates:
    acc, _ = evaluate(public_balls, X_public, y_public, temperature=t)
    print(f"  T={t:4.1f} → public accuracy = {acc*100:.2f}%")
    if acc > best_acc_temp:
        best_acc_temp = acc
        best_temp = t
print(f"\nBest temperature: {best_temp} (public accuracy {best_acc_temp*100:.2f}%)")

# ------------------------------------------------------------------
# 2. Epsilon sweep for DP‑GbC (only one seed to save time; you can increase)
# ------------------------------------------------------------------
epsilons = [0.5, 1.0, 2.0, 5.0, 10.0]
dp_gbc_acc = {}
print("\n--- DP‑GbC epsilon sweep (single seed, using best temperature) ---")
for eps in epsilons:
    dp_balls = apply_dp_to_balls(public_balls, epsilon_total=eps, delta_total=1e-5)
    acc, _ = evaluate(dp_balls, X_test, y_test, temperature=best_temp)
    dp_gbc_acc[eps] = acc
    print(f"  ε={eps:4.1f} → DP‑GbC test accuracy = {acc*100:.2f}%")

# ------------------------------------------------------------------
# 3. Epsilon sweep for DP‑SGD (single seed for speed; increase seeds if needed)
# ------------------------------------------------------------------
print("\n--- DP‑SGD epsilon sweep (single seed, target ε) ---")
dp_sgd_acc = {}
for target_eps in epsilons:
    model, eps_actual, acc = train_dp_sgd(target_epsilon=target_eps, epochs=4)
    dp_sgd_acc[target_eps] = (eps_actual, acc)
    print(f"  target ε={target_eps:4.1f} → actual ε={eps_actual:.2f}, test acc={acc*100:.2f}%")

# ------------------------------------------------------------------
# 4. Simple Membership Inference Attack (loss‑based threshold)
#    We use the distance‑based loss for GBC/DP‑GbC and cross‑entropy loss for DP‑SGD.
#    For simplicity, we train a threshold classifier on a small shadow set.
# ------------------------------------------------------------------
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression

def compute_loss_for_gbc(balls, X, y, temperature):
    losses = []
    centers = np.stack([b['center'] for b in balls])
    purities = np.array([b['purity'] for b in balls])
    for x, true_label in zip(X, y):
        dists = np.linalg.norm(centers - x, axis=1)
        dists_stable = dists - np.max(dists)
        weights = np.exp(np.clip(-dists_stable / temperature, -700, 700)) * purities
        if weights.sum() == 0:
            weights = np.ones_like(weights)
        class_scores = np.zeros(10)
        for i, b in enumerate(balls):
            class_scores[b['label']] += weights[i]
        probs = class_scores / class_scores.sum()
        loss = -np.log(probs[true_label] + 1e-12)
        losses.append(loss)
    return np.array(losses)

# Train a shadow model on half of the private set, evaluate on the other half
def mia_gbc(balls, X_train, y_train, X_test, y_test, temperature, seed=0):
    np.random.seed(seed)
    n = len(X_train)
    idx = np.random.choice(n, size=n//2, replace=False)
    mask = np.zeros(n, dtype=bool)
    mask[idx] = True
    # Compute losses for all training points (using the same balls)
    losses_all = compute_loss_for_gbc(balls, X_train, y_train, temperature)
    train_losses = losses_all[mask]
    test_losses  = losses_all[~mask]
    # Train a simple threshold (logistic regression) on one feature
    X_attack = np.concatenate([train_losses, test_losses]).reshape(-1,1)
    y_attack = np.concatenate([np.ones_like(train_losses), np.zeros_like(test_losses)])
    clf = LogisticRegression().fit(X_attack, y_attack)
    # Evaluate on the same set (in practice use separate shadow models, but for demonstration)
    # Instead, we evaluate on the same balls' losses (this is optimistic, but simple)
    pred_train = clf.predict(train_losses.reshape(-1,1))
    pred_test  = clf.predict(test_losses.reshape(-1,1))
    all_preds = np.concatenate([pred_train, pred_test])
    all_true  = np.concatenate([np.ones_like(train_losses), np.zeros_like(test_losses)])
    return accuracy_score(all_true, all_preds)

# For DP‑SGD, use cross‑entropy loss
def mia_sgd(model, X_train, y_train, X_test, y_test, seed=0):
    np.random.seed(seed)
    n = len(X_train)
    idx = np.random.choice(n, size=n//2, replace=False)
    mask = np.zeros(n, dtype=bool)
    mask[idx] = True
    # Compute cross‑entropy losses
    def ce_loss(model, X, y):
        model.eval()
        losses = []
        with torch.no_grad():
            for i in range(0, len(X), 256):
                xb = torch.from_numpy(X[i:i+256]).float().to(device)
                yb = torch.from_numpy(y[i:i+256]).long().to(device)
                logits = model(xb)
                loss = nn.functional.cross_entropy(logits, yb, reduction='none')
                losses.append(loss.cpu().numpy())
        return np.concatenate(losses)
    losses_all = ce_loss(model, X_train, y_train)
    train_losses = losses_all[mask]
    test_losses  = losses_all[~mask]
    X_attack = np.concatenate([train_losses, test_losses]).reshape(-1,1)
    y_attack = np.concatenate([np.ones_like(train_losses), np.zeros_like(test_losses)])
    clf = LogisticRegression().fit(X_attack, y_attack)
    pred_train = clf.predict(train_losses.reshape(-1,1))
    pred_test  = clf.predict(test_losses.reshape(-1,1))
    all_preds = np.concatenate([pred_train, pred_test])
    all_true  = np.concatenate([np.ones_like(train_losses), np.zeros_like(test_losses)])
    return accuracy_score(all_true, all_preds)

print("\n--- Membership Inference Attack (simple loss‑based) ---")
# For non‑private GBC (public balls)
mia_nonprivate = mia_gbc(public_balls, X_private, y_private, X_test, y_test, best_temp, seed=42)
print(f"Non‑private GBC MIA success = {mia_nonprivate*100:.2f}%")

# For DP‑GbC at ε=5
dp_balls = apply_dp_to_balls(public_balls, epsilon_total=5.0, delta_total=1e-5)
mia_dp_gbc = mia_gbc(dp_balls, X_private, y_private, X_test, y_test, best_temp, seed=42)
print(f"DP‑GbC (ε=5.0) MIA success = {mia_dp_gbc*100:.2f}%")

# For DP‑SGD (train one model at ε≈5)
sgd_model, eps_actual, _ = train_dp_sgd(target_epsilon=5.0, epochs=4)
mia_dp_sgd = mia_sgd(sgd_model, X_private, y_private, X_test, y_test, seed=42)
print(f"DP‑SGD (ε≈{eps_actual:.2f}) MIA success = {mia_dp_sgd*100:.2f}%")

# ------------------------------------------------------------------
# 5. Summary table of additional analyses
# ------------------------------------------------------------------
print("\n" + "="*70)
print("SUMMARY OF ADDITIONAL ANALYSES")
print("="*70)
print(f"Best temperature for GBC prediction: {best_temp}")
print("\nDP‑GbC accuracy vs ε (single seed):")
for eps, acc in dp_gbc_acc.items():
    print(f"  ε={eps:4.1f} → {acc*100:.2f}%")
print("\nDP‑SGD accuracy vs ε (single seed):")
for target, (eps_act, acc) in dp_sgd_acc.items():
    print(f"  target ε={target:4.1f} → actual ε={eps_act:.2f}, acc={acc*100:.2f}%")
print("\nMembership inference attack success (higher = more leakage):")
print(f"  Non‑private GBC: {mia_nonprivate*100:.2f}%")
print(f"  DP‑GbC (ε=5):    {mia_dp_gbc*100:.2f}%")
print(f"  DP‑SGD (ε≈{eps_actual:.2f}): {mia_dp_sgd*100:.2f}%")

In [ ]:
# ============================================================
# Additional experiments on simpler datasets (low‑dim / separable)
# ============================================================
print("\n" + "="*70)
print("TESTING DP‑GbC ON SIMPLER DATASETS (UCI Adult, MNIST, Two Moons)")
print("="*70)

from sklearn.datasets import fetch_openml, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd

# -------------------------------
# Helper to run full pipeline on a given dataset
# -------------------------------
def evaluate_on_dataset(X, y, dataset_name, public_ratio=0.2, test_ratio=0.2, epsilon=5.0, epochs=4):
    """
    X, y: numpy arrays.
    Splits into public (20%), private (60%), test (20%).
    Runs DP‑GbC and DP‑SGD on the private set, with balls built on public set.
    Returns acc_gbc, acc_sgd, eps_actual.
    """
    np.random.seed(42)
    n = len(X)
    idx = np.random.permutation(n)
    n_public = int(public_ratio * n)
    n_private = int(0.6 * n)
    public_idx = idx[:n_public]
    private_idx = idx[n_public:n_public+n_private]
    test_idx = idx[n_public+n_private:]

    X_public, y_public = X[public_idx], y[public_idx]
    X_private, y_private = X[private_idx], y[private_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # Scale features to [-1,1] using private statistics
    minv, maxv = X_private.min(axis=0), X_private.max(axis=0)
    rng = maxv - minv + 1e-8
    def scale(arr): return 2 * (arr - minv) / rng - 1.0
    X_public = scale(X_public)
    X_private = scale(X_private)
    X_test = scale(X_test)

    # Build balls on public set (using hyperparameters tuned for each dataset)
    # For low‑dim datasets, we can use default theta=0.1, min_samples=20
    public_balls = generate_granular_balls(X_public, y_public, theta=0.1, min_samples=20)

    # DP‑GbC
    dp_balls = apply_dp_to_balls(public_balls, epsilon_total=epsilon, delta_total=1e-5)
    acc_gbc, _ = evaluate(dp_balls, X_test, y_test, temperature=1.0)

    # DP‑SGD (on private data)
    # We need to convert to DataLoader
    private_feat_loader = make_feature_loader(X_private, y_private, shuffle=True)
    test_feat_loader = make_feature_loader(X_test, y_test, shuffle=False)

    # redefine train_dp_sgd to work with any feature loader (use global function but pass loaders)
    def train_dp_sgd_on_data(X_tr, y_tr, X_te, y_te, target_epsilon=epsilon, epochs=epochs):
        model = FCNet(input_dim=X_tr.shape[1]).to(device)
        model = ModuleValidator.fix(model)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
        criterion = nn.CrossEntropyLoss()
        batch_size = 256
        sample_rate = batch_size / len(X_tr)
        steps = epochs * (len(X_tr) // batch_size)
        noise_multiplier = get_noise_multiplier(
            target_epsilon=target_epsilon,
            target_delta=1e-5,
            sample_rate=sample_rate,
            steps=steps,
            accountant="rdp"
        )
        privacy_engine = PrivacyEngine()
        tr_loader = make_feature_loader(X_tr, y_tr, shuffle=True)
        te_loader = make_feature_loader(X_te, y_te, shuffle=False)
        model, optimizer, train_loader_dp = privacy_engine.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=tr_loader,
            noise_multiplier=noise_multiplier,
            max_grad_norm=1.0,
        )
        model.train()
        for ep in range(epochs):
            for xb, yb in train_loader_dp:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
        actual_epsilon = privacy_engine.get_epsilon(1e-5)
        model.eval()
        all_preds = []
        with torch.no_grad():
            for xb, _ in te_loader:
                xb = xb.to(device)
                preds = model(xb).argmax(dim=1).cpu().numpy()
                all_preds.append(preds)
        test_acc = accuracy_score(y_te, np.concatenate(all_preds))
        return actual_epsilon, test_acc

    eps_actual, acc_sgd = train_dp_sgd_on_data(X_private, y_private, X_test, y_test)

    print(f"{dataset_name}: DP‑GbC acc = {acc_gbc*100:.2f}%, DP‑SGD acc = {acc_sgd*100:.2f}% (ε≈{eps_actual:.2f})")
    return acc_gbc, acc_sgd, eps_actual

# -------------------------------
# 1. UCI Adult dataset (binary classification, 14 features after encoding)
# -------------------------------
print("\n--- UCI Adult (binary classification) ---")
adult = fetch_openml('adult', version=2, as_frame=True)
df = adult.frame
# Encode categorical features (simple: use pandas get_dummies)
df_encoded = pd.get_dummies(df, drop_first=True)
# Separate features and target (target is 'class' -> convert to 0/1)
y = (df_encoded['class_>50K']).astype(int).values
X = df_encoded.drop(columns=['class_>50K']).values.astype(float)
# Take a subset of 5000 samples for speed
np.random.seed(42)
idx = np.random.choice(len(X), 5000, replace=False)
X, y = X[idx], y[idx]
# Standardize numeric features (not necessary for scaling to [-1,1] later, but helps)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
evaluate_on_dataset(X, y, "Adult (binary)", epsilon=5.0)

# -------------------------------
# 2. MNIST (flattened 784-D, 10 classes) – use a subset of 5000 samples
# -------------------------------
print("\n--- MNIST (10 classes, 784-D) ---")
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_mnist = mnist.data.astype(np.float32) / 255.0  # scale to [0,1]
y_mnist = mnist.target.astype(int)
# Subset 5000 samples
np.random.seed(42)
idx = np.random.choice(len(X_mnist), 5000, replace=False)
X_mnist, y_mnist = X_mnist[idx], y_mnist[idx]
evaluate_on_dataset(X_mnist, y_mnist, "MNIST (784-D)", epsilon=5.0, epochs=4)

# -------------------------------
# 3. Synthetic Two Moons (binary, 2-D, separable)
# -------------------------------
print("\n--- Synthetic Two Moons (2-D, binary) ---")
X_moons, y_moons = make_moons(n_samples=2000, noise=0.1, random_state=42)
# Scale to [-1,1] already (values approx -1.5 to 1.5), we will let scaling handle it.
evaluate_on_dataset(X_moons, y_moons, "Two Moons (2-D)", epsilon=5.0, epochs=4)

# -------------------------------
# 4. Hybrid approach: use ball distances as features for DP‑SGD
#    (only on CIFAR‑10 for demonstration)
# -------------------------------
print("\n" + "="*70)
print("HYBRID: Use ball distances as features for DP‑SGD (on CIFAR‑10)")
print("="*70)
# Reuse existing public balls (from earlier best hyperparameters: 4 balls, min_samples=400)
# But we need to generate them again with the public set.
hybrid_balls = generate_granular_balls(X_public, y_public, theta=0.1, min_samples=400)
print(f"Number of balls: {len(hybrid_balls)}")
# Transform all data (private and test) to distance vectors to these balls
def distance_features(balls, X):
    centers = np.stack([b['center'] for b in balls])
    return np.linalg.norm(centers - X[:, np.newaxis, :], axis=2)  # shape (n_samples, n_balls)

X_private_dist = distance_features(hybrid_balls, X_private)
X_test_dist = distance_features(hybrid_balls, X_test)

# Now train DP‑SGD on these distance features (dim = n_balls = 4)
class SmallNet(nn.Module):
    def __init__(self, input_dim, n_classes=10):
        super().__init__()
        self.net = nn.Linear(input_dim, n_classes)  # simple linear
    def forward(self, x):
        return self.net(x)

def train_hybrid_sgd(X_tr, y_tr, X_te, y_te, target_epsilon=5.0, epochs=4):
    model = SmallNet(input_dim=X_tr.shape[1]).to(device)
    model = ModuleValidator.fix(model)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    batch_size = 256
    sample_rate = batch_size / len(X_tr)
    steps = epochs * (len(X_tr) // batch_size)
    noise_multiplier = get_noise_multiplier(
        target_epsilon=target_epsilon,
        target_delta=1e-5,
        sample_rate=sample_rate,
        steps=steps,
        accountant="rdp"
    )
    privacy_engine = PrivacyEngine()
    tr_loader = make_feature_loader(X_tr, y_tr, shuffle=True)
    te_loader = make_feature_loader(X_te, y_te, shuffle=False)
    model, optimizer, train_loader_dp = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=tr_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=1.0,
    )
    model.train()
    for ep in range(epochs):
        for xb, yb in train_loader_dp:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    actual_epsilon = privacy_engine.get_epsilon(1e-5)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in te_loader:
            xb = xb.to(device)
            preds = model(xb).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
    test_acc = accuracy_score(y_te, np.concatenate(all_preds))
    return actual_epsilon, test_acc

eps_hybrid, acc_hybrid = train_hybrid_sgd(X_private_dist, y_private, X_test_dist, y_test, target_epsilon=5.0, epochs=4)
print(f"Hybrid (distance features to 4 balls) → ε≈{eps_hybrid:.2f}, test acc = {acc_hybrid*100:.2f}%")
print("(Comparison: original DP‑SGD on raw features gave ~44.6%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle

# Generate synthetic data points for three classes
np.random.seed(42)
# Ball 1: class 0 (red) centered at (2,2)
points1 = np.random.randn(20,2) * 0.5 + [2,2]
# Ball 2: class 1 (blue) centered at (5,5)
points2 = np.random.randn(20,2) * 0.6 + [5,5]
# Ball 3: class 2 (green) centered at (8,3)
points3 = np.random.randn(20,2) * 0.4 + [8,3]

fig, ax = plt.subplots(figsize=(8,6))

# Plot data points
ax.scatter(points1[:,0], points1[:,1], c='red', s=50, label='Class 0', alpha=0.7)
ax.scatter(points2[:,0], points2[:,1], c='blue', s=50, label='Class 1', alpha=0.7)
ax.scatter(points3[:,0], points3[:,1], c='green', s=50, label='Class 2', alpha=0.7)

# Ball centers
centers = [(2,2), (5,5), (8,3)]
colors = ['red', 'blue', 'green']
radii = [1.2, 1.5, 0.9]

for (cx,cy), r, col in zip(centers, radii, colors):
    circle = Circle((cx,cy), r, fill=False, edgecolor=col, linewidth=2, linestyle='--')
    ax.add_patch(circle)
    ax.plot(cx, cy, marker='o', markersize=12, color=col, markeredgecolor='black', markerfacecolor=col)

ax.set_xlim(0,10)
ax.set_ylim(0,7)
ax.set_xlabel('Feature X', fontsize=12)
ax.set_ylabel('Feature Y', fontsize=12)
ax.set_title('Granular Balls: Data points grouped into spherical regions', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('granular_balls_concept.pdf', dpi=300)
plt.show()

In [ ]:
# Data from your experiments (approximate mean accuracies at ε≈4.5)
dimensions = [2, 14, 100, 512, 784]
dp_gbc_acc = [86.0, 75.8, 50.0, 16.8, 9.9]   # estimated 50% at 100D is a guess; adjust if you have real values
dp_sgd_acc = [87.8, 75.8, 60.0, 44.6, 74.5]  # Adult:75.8, CIFAR:44.6, MNIST:74.5

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(dimensions, dp_gbc_acc, 'o-', color='orange', linewidth=2, markersize=8, label='DP‑GbC')
ax.plot(dimensions, dp_sgd_acc, 's-', color='blue', linewidth=2, markersize=8, label='DP‑SGD')
ax.axvspan(2, 20, alpha=0.2, color='green', label='Works (d ≤ 20)')
ax.axvspan(100, 800, alpha=0.2, color='red', label='Fails (d ≥ 100)')
ax.set_xscale('log')
ax.set_xlabel('Feature Dimensionality (log scale)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('DP‑GbC vs. DP‑SGD: A Sharp Dimensionality Transition', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('dimensionality_cliff.pdf', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.path import Path

fig, ax = plt.subplots(figsize=(12,6))
ax.set_xlim(0,10)
ax.set_ylim(0,6)
ax.axis('off')

# Blocks
steps = [
    (1,5,"Public/Private\nSplit\n(1000 / 4000)"),
    (3,4,"Feature Extraction\n(ResNet-18, 512-D)"),
    (5,3,"Ball Generation\n(public data)"),
    (7,4,"DP Noise\n(per-ball, ε split)"),
    (9,5,"Evaluation\n(Accuracy, MIA)")
]
for x,y,text in steps:
    rect = patches.FancyBboxPatch((x-0.8, y-0.5), 1.6, 1.0,
                                  boxstyle="round,pad=0.1",
                                  facecolor='lightblue', edgecolor='black')
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=10)

# Arrows
arrows = [(1.5,4.5, 2.2,4.5), (3.8,4.5, 4.2,3.5), (5.8,3.5, 6.2,4.5), (7.8,4.5, 8.2,5.0)]
for x1,y1,x2,y2 in arrows:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.set_title('DP‑GbC Experimental Pipeline', fontsize=14)
plt.tight_layout()
plt.savefig('pipeline.pdf', dpi=300)
plt.show()